# Chinese BLEU Evaluation

This notebook follows the assignment template idea: segment Chinese sentences with `jieba`, then compute BLEU with `evaluate.load("bleu")`.

This file is used only to separately demonstrate the Part 1.2 requirement of implementing a BLEU score evaluation script. It is not the model training script; training is implemented in the corresponding translator Python files and in `part1_machine_translation.ipynb`.

Workflow note: `translator_en2cn_baseline.py` trains the baseline model from scratch, evaluates dev BLEU every 5 epochs, and finally writes test-set translations to `baseline_test_predictions.txt`; the baseline test BLEU is then computed here from that saved prediction file. In contrast, `translator_en2cn_cosine.py` already computes and saves the final test BLEU directly after training.

Expected input is the tab-separated prediction text file generated by `translator_en2cn_baseline.py`:

```text
outputs/part1_machine_translation/baseline/baseline_test_predictions.txt
```

The text file should contain at least `reference` and `prediction` columns.

In [1]:
# If the environment does not have these packages, install them first:
# !pip install jieba evaluate


In [2]:
from pathlib import Path
import csv
import json

import jieba
import evaluate


In [3]:
# Project paths
NOTEBOOK_DIR = Path.cwd()
ASSIGNMENT_ROOT = NOTEBOOK_DIR.parents[1]

prediction_file = ASSIGNMENT_ROOT / "outputs" / "part1_machine_translation" / "baseline" / "baseline_test_predictions.txt"
result_file = ASSIGNMENT_ROOT / "outputs" / "part1_machine_translation" / "baseline" / "baseline_bleu_result.json"

print("prediction_file:", prediction_file)
print("result_file:", result_file)


prediction_file: E:\code\DPfinal2PG\12533582_assignment2\outputs\part1_machine_translation\baseline\baseline_test_predictions.txt
result_file: E:\code\DPfinal2PG\12533582_assignment2\outputs\part1_machine_translation\baseline\baseline_bleu_result.json


In [4]:
def segment_chinese(sentence: str) -> str:
    """Segment one Chinese sentence into a whitespace-separated string for BLEU."""
    words = list(jieba.cut(sentence.strip(), cut_all=False))
    return " ".join(word for word in words if word.strip())


def load_prediction_file(path: Path):
    """Load references and predictions from tab-separated translator output text."""
    predictions = []
    references = []
    examples = []

    with path.open("r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f, delimiter="\t")
        for row in reader:
            prediction = row.get("prediction", "").strip()
            reference = row.get("reference", "").strip()

            # Fallback for files that only store segmented tokens.
            if not prediction and row.get("prediction_tokens"):
                prediction = row["prediction_tokens"].replace(" ", "")
            if not reference and row.get("reference_tokens"):
                reference = row["reference_tokens"].replace(" ", "")

            if not prediction or not reference:
                continue

            segmented_prediction = segment_chinese(prediction)
            segmented_reference = segment_chinese(reference)

            predictions.append(segmented_prediction)
            references.append([segmented_reference])
            examples.append({
                "source": row.get("source", ""),
                "reference": reference,
                "prediction": prediction,
                "reference_segmented": segmented_reference,
                "prediction_segmented": segmented_prediction,
            })

    return predictions, references, examples


In [5]:
if not prediction_file.exists():
    raise FileNotFoundError(
        f"Prediction file not found: {prediction_file}\n"
        "Run translator_en2cn_baseline.py first to generate test predictions."
    )

predictions, references, examples = load_prediction_file(prediction_file)
print("Loaded examples:", len(predictions))

for item in examples[:3]:
    print("-" * 80)
    print("SOURCE:", item["source"])
    print("REFERENCE:", item["reference"])
    print("PREDICTION:", item["prediction"])
    print("REFERENCE SEG:", item["reference_segmented"])
    print("PREDICTION SEG:", item["prediction_segmented"])


Building prefix dict from the default dictionary ...


Loading model from cache C:\Users\duan\AppData\Local\Temp\jieba.cache


Loading model cost 0.780 seconds.


Prefix dict has been built successfully.


Loaded examples: 1817
--------------------------------------------------------------------------------
SOURCE: UNK !
REFERENCE: 干杯!
PREDICTION: 先生了。
REFERENCE SEG: 干杯 !
PREDICTION SEG: 先生 了 。
--------------------------------------------------------------------------------
SOURCE: listen .
REFERENCE: 听着。
PREDICTION: 听听听。
REFERENCE SEG: 听 着 。
PREDICTION SEG: 听听 听 。
--------------------------------------------------------------------------------
SOURCE: try it .
REFERENCE: 试试吧。
PREDICTION: 试试试。
REFERENCE SEG: 试试 吧 。
PREDICTION SEG: 试试 试 。


In [6]:
bleu = evaluate.load("bleu")
results = bleu.compute(predictions=predictions, references=references)

print(results)

result_file.parent.mkdir(parents=True, exist_ok=True)
with result_file.open("w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print("Saved BLEU result to:", result_file)


{'bleu': 0.16738774604220316, 'precisions': [0.5414767547857794, 0.2231219512195122, 0.1152614727854856, 0.06341537067794051], 'brevity_penalty': 0.9710094516335082, 'length_ratio': 0.9714216712284657, 'translation_length': 12067, 'reference_length': 12422}
Saved BLEU result to: E:\code\DPfinal2PG\12533582_assignment2\outputs\part1_machine_translation\baseline\baseline_bleu_result.json
